# NFL Player Contact Detection: Tracking Feature Model

This notebook improves on the distance-only baseline with an offline-safe tabular model. It learns from player tracking features for both player-player and player-ground rows, tunes a probability threshold with MCC, and writes `submission.csv`.


## 1. Setup and Configuration

The model is intentionally Kaggle-safe: no internet, no external packages, grouped validation by `game_play`, controlled negative sampling, and a single submission output.


In [1]:
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import matthews_corrcoef, precision_recall_fscore_support
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

NOTEBOOK_VERSION = "TRACKING_FEATURE_V5_DYNAMICS"
NOTEBOOK_NAME = "Tracking Feature Model"
RANDOM_STATE = 42
TARGET = "contact"
ID_COL = "contact_id"
RUN_FAST = False
FAST_SAMPLE_PLAYS = 35
NEGATIVE_TO_POSITIVE_RATIO = 6
MAX_TRAIN_ROWS = 650_000
VALID_SIZE = 0.25
THRESHOLDS = np.round(np.arange(0.05, 0.96, 0.01), 2)

REQUIRED_DATA_FILES = [
    "train_labels.csv",
    "sample_submission.csv",
    "train_player_tracking.csv",
    "test_player_tracking.csv",
]

DATA_DIR = Path("/kaggle/input/competitions/nfl-player-contact-detection")
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

missing_files = [
    filename for filename in REQUIRED_DATA_FILES
    if not (DATA_DIR / filename).exists()
]
if missing_files:
    raise FileNotFoundError(
        f"Missing required files in {DATA_DIR}: {missing_files}"
    )

print(f"NOTEBOOK_VERSION: {NOTEBOOK_VERSION}")
print(f"NOTEBOOK_NAME: {NOTEBOOK_NAME}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR.resolve()}")


NOTEBOOK_VERSION: TRACKING_FEATURE_V5_DYNAMICS
NOTEBOOK_NAME: Tracking Feature Model
DATA_DIR: /kaggle/input/competitions/nfl-player-contact-detection
OUTPUT_DIR: /kaggle/working


### Version Check

When this notebook runs on Kaggle, the first code cell prints `NOTEBOOK_VERSION = TRACKING_FEATURE_V5_DYNAMICS`. If Kaggle shows an older version string, pull/sync the latest notebook before trusting the output.


## 2. Helper Functions

Feature construction now includes prior-step tracking dynamics, cyclic direction/orientation encodings, pair distance, pair-distance change, relative motion, angular differences, and same-team flags. These features give the model contact-onset signals that the static distance baseline cannot see.


In [2]:
def reduce_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast dataframe columns to reduce notebook memory usage."""
    out = df.copy()
    for col in out.columns:
        dtype = out[col].dtype
        if pd.api.types.is_integer_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="integer")
        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")
        elif pd.api.types.is_object_dtype(dtype):
            nunique = out[col].nunique(dropna=False)
            if nunique / max(len(out), 1) < 0.5 and col != ID_COL:
                out[col] = out[col].astype("category")
    return out


def parse_contact_id(df: pd.DataFrame) -> pd.DataFrame:
    """Split Kaggle contact_id into play, step, player1, and player2 fields."""
    out = df.copy()
    parts = out[ID_COL].astype(str).str.split("_", expand=True)
    out["game_play"] = parts[0] + "_" + parts[1]
    out["step"] = parts[2].astype("int16")
    out["nfl_player_id_1"] = parts[3].astype(str)
    out["nfl_player_id_2"] = parts[4].astype(str)
    out["is_ground"] = out["nfl_player_id_2"].eq("G").astype("int8")
    out["contact_type"] = np.where(out["is_ground"].eq(1), "ground", "player_player")
    return out


def build_category_maps(*tracking_frames: pd.DataFrame) -> dict[str, dict[str, int]]:
    """Build stable integer maps for low-cardinality tracking categories."""
    maps = {}
    for col in ["position", "team"]:
        values = []
        for frame in tracking_frames:
            if col in frame.columns:
                values.extend(frame[col].astype(str).fillna("missing").unique().tolist())
        maps[col] = {value: idx for idx, value in enumerate(sorted(set(values)))}
    return maps


def prepare_tracking(tracking: pd.DataFrame, category_maps: dict[str, dict[str, int]]) -> pd.DataFrame:
    """Select, encode, and add prior-step tracking dynamics."""
    keep_cols = [
        "game_play", "step", "nfl_player_id", "x_position", "y_position",
        "speed", "distance", "direction", "orientation", "acceleration", "sa",
        "position", "team", "jersey_number",
    ]
    keep_cols = [col for col in keep_cols if col in tracking.columns]
    out = tracking[keep_cols].copy()
    out["nfl_player_id"] = out["nfl_player_id"].astype(str)
    out = out.sort_values(["game_play", "nfl_player_id", "step"]).reset_index(drop=True)

    for angle_col in ["direction", "orientation"]:
        if angle_col in out.columns:
            radians = np.deg2rad(out[angle_col])
            out[f"{angle_col}_sin"] = np.sin(radians).astype("float32")
            out[f"{angle_col}_cos"] = np.cos(radians).astype("float32")

    group = out.groupby(["game_play", "nfl_player_id"], observed=True)
    for col in ["x_position", "y_position", "speed", "distance", "acceleration", "sa"]:
        if col in out.columns:
            lag_col = f"{col}_lag1"
            out[lag_col] = group[col].shift(1)
            out[f"{col}_delta1"] = out[col] - out[lag_col]

    for col in ["direction", "orientation"]:
        if col in out.columns:
            lag = group[col].shift(1)
            diff = (out[col] - lag).abs() % 360
            out[f"{col}_delta1"] = np.minimum(diff, 360 - diff)

    for col, mapping in category_maps.items():
        if col in out.columns:
            out[f"{col}_code"] = (
                out[col].astype(str).fillna("missing").map(mapping).fillna(-1).astype("int16")
            )
        else:
            out[f"{col}_code"] = -1
    drop_cols = [col for col in ["position", "team"] if col in out.columns]
    return out.drop(columns=drop_cols)


def angular_difference(left: pd.Series, right: pd.Series) -> pd.Series:
    """Return smallest absolute angular difference in degrees."""
    diff = (left - right).abs() % 360
    return np.minimum(diff, 360 - diff)


def attach_tracking_features(contacts: pd.DataFrame, tracking: pd.DataFrame) -> pd.DataFrame:
    """Attach player tracking features and pairwise relative features."""
    base = contacts.copy()
    track = tracking.copy()
    player_cols = [
        col for col in track.columns
        if col not in ["game_play", "step", "nfl_player_id"]
    ]

    out = base.merge(
        track,
        left_on=["game_play", "step", "nfl_player_id_1"],
        right_on=["game_play", "step", "nfl_player_id"],
        how="left",
    ).drop(columns=["nfl_player_id"])
    out = out.rename(columns={col: f"p1_{col}" for col in player_cols})

    p2_rows = base[base["is_ground"].eq(0)].copy()
    p2 = p2_rows[[ID_COL, "game_play", "step", "nfl_player_id_2"]].merge(
        track,
        left_on=["game_play", "step", "nfl_player_id_2"],
        right_on=["game_play", "step", "nfl_player_id"],
        how="left",
    ).drop(columns=["game_play", "step", "nfl_player_id_2", "nfl_player_id"])
    p2 = p2.rename(columns={col: f"p2_{col}" for col in player_cols})
    out = out.merge(p2, on=ID_COL, how="left")

    out["dx"] = out["p1_x_position"] - out["p2_x_position"]
    out["dy"] = out["p1_y_position"] - out["p2_y_position"]
    out["pair_distance"] = np.sqrt(np.square(out["dx"]) + np.square(out["dy"]))
    out["abs_dx"] = out["dx"].abs()
    out["abs_dy"] = out["dy"].abs()

    lag_cols = [
        "p1_x_position_lag1", "p2_x_position_lag1",
        "p1_y_position_lag1", "p2_y_position_lag1",
    ]
    if all(col in out.columns for col in lag_cols):
        out["dx_lag1"] = out["p1_x_position_lag1"] - out["p2_x_position_lag1"]
        out["dy_lag1"] = out["p1_y_position_lag1"] - out["p2_y_position_lag1"]
        out["pair_distance_lag1"] = np.sqrt(
            np.square(out["dx_lag1"]) + np.square(out["dy_lag1"])
        )
        out["pair_distance_delta1"] = out["pair_distance"] - out["pair_distance_lag1"]
        out["abs_pair_distance_delta1"] = out["pair_distance_delta1"].abs()

    for col in ["speed", "distance", "acceleration", "sa"]:
        left = f"p1_{col}"
        right = f"p2_{col}"
        if left in out.columns and right in out.columns:
            out[f"{col}_diff"] = out[left] - out[right]
            out[f"abs_{col}_diff"] = out[f"{col}_diff"].abs()

    for col in ["direction", "orientation"]:
        left = f"p1_{col}"
        right = f"p2_{col}"
        if left in out.columns and right in out.columns:
            out[f"{col}_angle_diff"] = angular_difference(out[left], out[right])

    if "p1_team_code" in out.columns and "p2_team_code" in out.columns:
        out["same_team"] = (out["p1_team_code"] == out["p2_team_code"]).astype("float32")

    out["step_float"] = out["step"].astype("float32")
    return out


def sample_training_rows(labels: pd.DataFrame) -> pd.DataFrame:
    """Keep all positives and a controlled number of negatives for training."""
    if RUN_FAST:
        labels = labels[labels["game_play"].isin(
            pd.Series(labels["game_play"].unique()).sample(
                min(FAST_SAMPLE_PLAYS, labels["game_play"].nunique()),
                random_state=RANDOM_STATE,
            )
        )].copy()

    positives = labels[labels[TARGET].eq(1)]
    negatives = labels[labels[TARGET].eq(0)]
    max_negatives = min(
        len(negatives),
        max(len(positives) * NEGATIVE_TO_POSITIVE_RATIO, MAX_TRAIN_ROWS - len(positives)),
    )
    negatives = negatives.sample(max_negatives, random_state=RANDOM_STATE)
    sampled = pd.concat([positives, negatives], ignore_index=True)
    if len(sampled) > MAX_TRAIN_ROWS:
        sampled = sampled.sample(MAX_TRAIN_ROWS, random_state=RANDOM_STATE)
    return sampled.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)


def get_feature_columns(df: pd.DataFrame) -> list[str]:
    """Return numeric model features, excluding IDs and target columns."""
    excluded = {
        ID_COL, TARGET, "game_play", "nfl_player_id_1", "nfl_player_id_2",
        "contact_type", "datetime",
    }
    return [
        col for col in df.columns
        if col not in excluded and pd.api.types.is_numeric_dtype(df[col])
    ]


def score_thresholds(y_true: np.ndarray, probabilities: np.ndarray) -> pd.DataFrame:
    """Score probability thresholds with MCC and positive rate."""
    rows = []
    for threshold in THRESHOLDS:
        pred = (probabilities >= threshold).astype("int8")
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, pred, average="binary", zero_division=0
        )
        rows.append({
            "threshold": threshold,
            "mcc": matthews_corrcoef(y_true, pred),
            "positive_rate": pred.mean(),
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })
    return pd.DataFrame(rows).sort_values("mcc", ascending=False)


def make_model():
    """Create the strongest offline-safe sklearn model available."""
    try:
        return HistGradientBoostingClassifier(
            learning_rate=0.06,
            max_iter=240,
            max_leaf_nodes=31,
            l2_regularization=0.05,
            early_stopping=True,
            validation_fraction=0.12,
            random_state=RANDOM_STATE,
        )
    except TypeError:
        return RandomForestClassifier(
            n_estimators=240,
            min_samples_leaf=8,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )


## 3. Load Data


In [3]:
labels = reduce_memory_usage(pd.read_csv(
    DATA_DIR / "train_labels.csv",
    parse_dates=["datetime"],
))
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
train_tracking = reduce_memory_usage(pd.read_csv(
    DATA_DIR / "train_player_tracking.csv",
    parse_dates=["datetime"],
))
test_tracking = reduce_memory_usage(pd.read_csv(
    DATA_DIR / "test_player_tracking.csv",
    parse_dates=["datetime"],
))

labels = parse_contact_id(labels)
sample_submission = parse_contact_id(sample_submission)
category_maps = build_category_maps(train_tracking, test_tracking)
train_tracking_prepared = prepare_tracking(train_tracking, category_maps)
test_tracking_prepared = prepare_tracking(test_tracking, category_maps)

print(f"labels: {labels.shape}")
print(f"sample_submission: {sample_submission.shape}")
print(f"train_tracking: {train_tracking_prepared.shape}")
print(f"test_tracking: {test_tracking_prepared.shape}")
print(f"positive rate: {labels[TARGET].mean():.5f}")
labels.head()


labels: (4721618, 9)
sample_submission: (49588, 8)
train_tracking: (1353053, 32)
test_tracking: (14872, 32)
positive rate: 0.01367


,contact_id,game_play,datetime,step,nfl_player_id_1,nfl_player_id_2,contact,is_ground,contact_type
0,58168_003392_0_38590_43854,58168_003392,2020-09-11 03:01:48.100000+00:00,0,38590,43854,0,0,player_player
1,58168_003392_0_38590_41257,58168_003392,2020-09-11 03:01:48.100000+00:00,0,38590,41257,0,0,player_player
2,58168_003392_0_38590_41944,58168_003392,2020-09-11 03:01:48.100000+00:00,0,38590,41944,0,0,player_player
3,58168_003392_0_38590_42386,58168_003392,2020-09-11 03:01:48.100000+00:00,0,38590,42386,0,0,player_player
4,58168_003392_0_38590_47944,58168_003392,2020-09-11 03:01:48.100000+00:00,0,38590,47944,0,0,player_player


## 4. Grouped Split and Feature Construction

Split by `game_play` before downsampling. The model trains on all positives plus sampled negatives from the training plays, while validation keeps the natural held-out play distribution for realistic MCC threshold tuning.


In [4]:

model_source = labels.copy()
if RUN_FAST:
    model_source = model_source[model_source["game_play"].isin(
        pd.Series(model_source["game_play"].unique()).sample(
            min(FAST_SAMPLE_PLAYS, model_source["game_play"].nunique()),
            random_state=RANDOM_STATE,
        )
    )].copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=VALID_SIZE,
    random_state=RANDOM_STATE,
)
train_source_idx, valid_source_idx = next(
    splitter.split(model_source, model_source[TARGET], groups=model_source["game_play"])
)
train_source = model_source.iloc[train_source_idx].copy()
valid_rows = model_source.iloc[valid_source_idx].copy()
train_rows = sample_training_rows(train_source)

print(f"source rows: {model_source.shape}")
print(f"training-source rows: {train_source.shape}")
print(f"sampled training rows: {train_rows.shape}")
print(f"natural validation rows: {valid_rows.shape}")
print(f"sampled training positive rate: {train_rows[TARGET].mean():.5f}")
print(f"natural validation positive rate: {valid_rows[TARGET].mean():.5f}")
display(train_rows.groupby("contact_type", observed=True)[TARGET].agg(["count", "mean", "sum"]))
display(valid_rows.groupby("contact_type", observed=True)[TARGET].agg(["count", "mean", "sum"]))

train_features = attach_tracking_features(train_rows, train_tracking_prepared)
valid_features = attach_tracking_features(valid_rows, train_tracking_prepared)
feature_cols = get_feature_columns(train_features)

print(f"train feature frame: {train_features.shape}")
print(f"valid feature frame: {valid_features.shape}")
print(f"feature count: {len(feature_cols)}")
display(train_features[feature_cols].head())

del model_source, train_source, train_rows, valid_rows
gc.collect()


source rows: (4721618, 9)
training-source rows: (3578311, 9)
sampled training rows: (650000, 9)
natural validation rows: (1143307, 9)
sampled training positive rate: 0.07763
natural validation positive rate: 0.01230


,count,mean,sum
contact_type,,,
ground,63748,0.209654,13365
player_player,586252,0.063278,37097


,count,mean,sum
contact_type,,,
ground,99418,0.034692,3449
player_player,1043889,0.010165,10611


train feature frame: (650000, 89)
valid feature frame: (1143307, 89)
feature count: 82


,step,is_ground,p1_x_position,p1_y_position,p1_speed,p1_distance,p1_direction,p1_orientation,p1_acceleration,p1_sa,p1_jersey_number,p1_direction_sin,p1_direction_cos,p1_orientation_sin,p1_orientation_cos,p1_x_position_lag1,p1_x_position_delta1,p1_y_position_lag1,p1_y_position_delta1,p1_speed_lag1,p1_speed_delta1,p1_distance_lag1,p1_distance_delta1,p1_acceleration_lag1,p1_acceleration_delta1,p1_sa_lag1,p1_sa_delta1,p1_direction_delta1,p1_orientation_delta1,p1_position_code,p1_team_code,p2_x_position,p2_y_position,p2_speed,p2_distance,p2_direction,p2_orientation,p2_acceleration,p2_sa,p2_jersey_number,p2_direction_sin,p2_direction_cos,p2_orientation_sin,p2_orientation_cos,p2_x_position_lag1,p2_x_position_delta1,p2_y_position_lag1,p2_y_position_delta1,p2_speed_lag1,p2_speed_delta1,p2_distance_lag1,p2_distance_delta1,p2_acceleration_lag1,p2_acceleration_delta1,p2_sa_lag1,p2_sa_delta1,p2_direction_delta1,p2_orientation_delta1,p2_position_code,p2_team_code,dx,dy,pair_distance,abs_dx,abs_dy,dx_lag1,dy_lag1,pair_distance_lag1,pair_distance_delta1,abs_pair_distance_delta1,speed_diff,abs_speed_diff,distance_diff,abs_distance_diff,acceleration_diff,abs_acceleration_diff,sa_diff,abs_sa_diff,direction_angle_diff,orientation_angle_diff,same_team,step_float
0,29,0,81.379997,30.260000,3.83,0.40,73.250000,316.779999,4.19,-4.03,19,0.957571,0.288196,-0.684802,0.728730,80.989998,0.389999,30.150000,0.110001,4.26,-0.43,0.45,-0.05,4.20,-0.01,-4.06,0.03,1.639999,29.329987,27,0,97.430000,34.259998,2.67,0.28,119.430000,181.740005,1.34,-1.24,51.0,0.870957,-0.491360,-0.030364,-0.999539,97.190002,0.239998,34.389999,-0.130001,2.76,-0.09,0.29,-0.01,1.33,0.01,-1.23,-0.01,1.559998,16.029999,3.0,0.0,-16.050003,-3.999998,16.540937,16.050003,3.999998,-16.200005,-4.240000,16.745678,-0.204741,0.204741,1.16,1.16,0.12,0.12,2.85,2.85,-2.79,2.79,46.180000,135.039993,1.0,29.0
1,41,0,68.370003,25.830000,3.13,0.30,326.230011,287.450012,2.73,2.67,93,-0.555860,0.831276,-0.953979,0.299874,68.540001,-0.169998,25.580000,0.250000,2.65,0.48,0.26,0.04,2.94,-0.21,2.84,-0.17,2.130005,5.770020,5,1,63.459999,30.260000,1.93,0.19,55.040001,168.080002,1.09,0.60,2.0,0.819552,0.573004,0.206546,-0.978437,63.310001,0.149998,30.139999,0.120001,1.80,0.13,0.18,0.01,1.06,0.03,0.41,0.19,3.480000,0.619995,20.0,0.0,4.910004,-4.430000,6.613096,4.910004,4.430000,5.230000,-4.559999,6.938767,-0.325671,0.325671,1.20,1.20,0.11,0.11,1.64,1.64,2.07,2.07,88.809998,119.370010,0.0,41.0
2,8,0,63.320000,28.670000,1.68,0.17,39.959999,109.660004,2.31,1.11,94,0.642253,0.766493,0.941706,-0.336438,63.220001,0.099998,28.530001,0.139999,1.51,0.17,0.15,0.02,1.34,0.97,0.64,0.47,14.400002,3.659996,3,0,62.650002,24.209999,2.91,0.30,4.550000,92.269997,1.20,0.97,99.0,0.079329,0.996848,0.999215,-0.039609,62.619999,0.030003,23.910000,0.299999,2.63,0.28,0.26,0.04,1.29,-0.09,1.12,-0.15,1.930000,0.000000,5.0,0.0,0.669998,4.460001,4.510045,0.669998,4.460001,0.600002,4.620001,4.658799,-0.148754,0.148754,-1.23,1.23,-0.13,0.13,1.11,1.11,0.14,0.14,35.410000,17.390007,1.0,8.0
3,35,0,44.310001,19.440001,4.26,0.44,177.639999,229.479996,1.48,-0.89,95,0.041178,-0.999152,-0.760179,-0.649714,44.290001,0.020000,19.879999,-0.439999,4.23,0.03,0.43,0.01,1.35,0.13,-0.90,0.01,2.449997,0.000000,5,1,43.669998,18.540001,3.74,0.38,167.419998,176.699997,0.80,-0.49,73.0,0.217803,-0.975993,0.057564,-0.998342,43.590000,0.079998,18.910000,-0.369999,3.76,-0.02,0.38,0.00,0.56,0.24,-0.50,0.01,2.309998,3.869995,16.0,0.0,0.640003,0.900000,1.104357,0.640003,0.900000,0.700001,0.969999,1.196202,-0.091846,0.091846,0.52,0.52,0.06,0.06,0.68,0.68,-0.40,0.40,10.220001,52.779999,0.0,35.0
4,15,0,26.530001,25.350000,5.47,0.55,116.400002,25.389999,0.18,-0.14,51,0.895712,-0.444635,0.428777,0.903410,26.040001,0.490000,25.600000,-0.250000,5.45,0.02,0.54,0.01,0.58,-0.40,0.18,-0.32,0.769997,7.690002,10,1,18.480000,33.430000,6.45,0.64,248.759995,230.929993,1.19,0.89,81.0,-0.932071,-0.362275,-0.776376,-0.630270,19.080000,-0.600000,33.660000,-0.230000,6.25,0.20,0.62,0.02,1.26,-0

28

## 5. Train/Validation Matrices

Prepare numeric matrices for the model. Missing player-2 values are expected for ground-contact rows and are handled by histogram gradient boosting.


In [5]:

X_train = train_features[feature_cols].replace([np.inf, -np.inf], np.nan)
y_train = train_features[TARGET].astype("int8")
X_valid = valid_features[feature_cols].replace([np.inf, -np.inf], np.nan)
y_valid = valid_features[TARGET].astype("int8")
valid_meta = valid_features[[ID_COL, "game_play", "contact_type"]].copy()

print(f"train rows: {len(X_train):,}")
print(f"valid rows: {len(X_valid):,}")
print(f"train positive rate: {y_train.mean():.5f}")
print(f"valid positive rate: {y_valid.mean():.5f}")


train rows: 650,000
valid rows: 1,143,307
train positive rate: 0.07763
valid positive rate: 0.01230


## 6. Train Model and Tune Threshold

The model outputs probabilities. MCC is optimized by searching hard-label thresholds on held-out plays.


In [6]:
model = make_model()
model.fit(X_train, y_train)
valid_prob = model.predict_proba(X_valid)[:, 1]
threshold_scores = score_thresholds(y_valid.to_numpy(), valid_prob)
best_threshold = float(threshold_scores.iloc[0]["threshold"])

print(f"model: {type(model).__name__}")
print(f"best threshold: {best_threshold:.2f}")
display(threshold_scores.head(12))

valid_pred = (valid_prob >= best_threshold).astype("int8")
valid_scored = valid_meta.assign(
    y_true=y_valid.to_numpy(),
    probability=valid_prob,
    prediction=valid_pred,
)
slice_scores = []
for contact_type, part in valid_scored.groupby("contact_type", observed=True):
    slice_scores.append({
        "contact_type": contact_type,
        "rows": len(part),
        "actual_rate": part["y_true"].mean(),
        "predicted_rate": part["prediction"].mean(),
        "mcc": matthews_corrcoef(part["y_true"], part["prediction"]),
    })
display(pd.DataFrame(slice_scores))


model: HistGradientBoostingClassifier
best threshold: 0.67


,threshold,mcc,positive_rate,precision,recall,f1
62,0.67,0.653104,0.014942,0.596734,0.725036,0.654658
64,0.69,0.652854,0.014253,0.610678,0.707752,0.655642
63,0.68,0.652722,0.014596,0.603368,0.716145,0.654937
61,0.66,0.652512,0.015288,0.589450,0.732788,0.653350
65,0.70,0.652435,0.013871,0.618576,0.697724,0.655771
66,0.71,0.652432,0.013522,0.626455,0.688834,0.656165
67,0.72,0.651866,0.013211,0.633210,0.680228,0.655877
60,0.65,0.651423,0.015620,0.582227,0.739545,0.651524
68,0.73,0.651276,0.012887,0.640491,0.671195,0.655484
59,0.64,0.651206,0.015964,0.575773,0.747440,0.650470


,contact_type,rows,actual_rate,predicted_rate,mcc
0,ground,99418,0.034692,0.029049,0.412314
1,player_player,1043889,0.010165,0.013598,0.716558


## 7. Train Final Model and Create Submission

Refit on the sampled training rows, score every submission row, and write `submission.csv`.


In [7]:

final_train_rows = sample_training_rows(labels)
final_features = attach_tracking_features(final_train_rows, train_tracking_prepared)
X_full = final_features[feature_cols].replace([np.inf, -np.inf], np.nan)
y_full = final_features[TARGET].astype("int8")
final_model = make_model()
final_model.fit(X_full, y_full)

submission_rows = sample_submission[[
    ID_COL, "game_play", "step", "nfl_player_id_1", "nfl_player_id_2",
    "is_ground", "contact_type",
]].copy()
test_features = attach_tracking_features(submission_rows, test_tracking_prepared)
X_test = test_features[feature_cols].replace([np.inf, -np.inf], np.nan)
test_prob = final_model.predict_proba(X_test)[:, 1]

submission = sample_submission[[ID_COL]].copy()
submission[TARGET] = (test_prob >= best_threshold).astype("int8")
output_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(output_path, index=False)

print(f"wrote: {output_path}")
print(f"submission shape: {submission.shape}")
print(f"predicted positive rate: {submission[TARGET].mean():.5f}")
display(submission.head())

del final_train_rows, final_features, X_full, X_test, test_features
gc.collect()


wrote: /kaggle/working/submission.csv
submission shape: (49588, 2)
predicted positive rate: 0.02025


,contact_id,contact
0,58168_003392_0_38590_43854,0
1,58168_003392_0_38590_41257,0
2,58168_003392_0_38590_41944,0
3,58168_003392_0_38590_42386,0
4,58168_003392_0_38590_47944,0


0

## 8. Next Improvements

This model should beat the pure distance baseline because it can predict ground rows and use movement dynamics. Next upgrades should add nearest-player context, helmet box visibility/size features, video-derived posture cues, and post-threshold smoothing validated by play-grouped OOF predictions.
